# La Plata: cargos ejecutivos, 2011-2023 (Presidente, Gobernador, Intendente)

La Plata, Provincia de Buenos Aires, Generales, para los tres cargos ejecutivos — **Presidente**, **Gobernador** e **Intendente** — a lo largo de cuatro elecciones: **2011, 2015, 2019 y 2023**. Cubre todo el ciclo: traer el CSV oficial de cada (año, categoría), ver el resultado, validarlo contra el agregado JSON, un ejemplo de análisis mesa por mesa, y confirmar el estado final del caché en disco.

Usamos `resultado/totalizadocsv` — el endpoint que arma el CSV oficial descargable del sitio (`GET /api/resultado/totalizadocsv`, con parámetros `año`, `recuento`, `idEleccion`, `idCargo`, `idDistrito`, `idSeccionProvincial`, `idSeccion`). Trae todas las mesas de la categoría en un solo pedido.

El `categoriaId`/`idCargo` de cada categoría se resolvió probando valores y leyendo el campo `cargo_nombre` que devuelve el propio CSV (no hay forma de derivarlo de otra manera). Para La Plata, el mapeo resultó **estable en los cuatro años**:

| categoriaId | cargo |
|---|---|
| 1 | PRESIDENTE |
| 4 | GOBERNADOR |
| 7 | INTENDENTE |


El caché de Generales queda en `data/<año>/<cargo>/generales/` (hermano de `paso/` y `balotaje/`, agregados en la sección 7-9 de este notebook — §2.3 del plan de correcciones).

In [1]:
import io
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from electoral.client import ResultadosClient
from electoral.models import ResultadoElectoral

REPO = Path.cwd().parent
client = ResultadosClient(cache_dir=REPO / "data" / "distrito")

ANIOS = [2011, 2015, 2019, 2023]

CATEGORIAS = {
    "presidente": 1,
    "gobernador": 4,
    "intendente": 7,
}

LA_PLATA = dict(
    tipo_eleccion=2,  # Generales
    distrito_id=2,  # Buenos Aires
    seccion_provincial_id=8,  # Sección Capital
    seccion_id=63,  # La Plata
)

## 1. Traer el CSV oficial de cada (año, categoría)

In [2]:
dataframes = {}
for anio in ANIOS:
    for nombre, categoria_id in CATEGORIAS.items():
        csv_bytes = client.get_resultados_csv(
            anio_eleccion=anio, categoria_nombre=f"{nombre}/generales", categoria_id=categoria_id, **LA_PLATA
        )
        df = pd.read_csv(io.BytesIO(csv_bytes))
        df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()
        dataframes[(anio, nombre)] = df
        print(f"{anio}/{nombre}: {len(df)} filas, {df['mesa_id'].nunique()} mesas, "
              f"mesa_tipo={sorted(df['mesa_tipo'].unique())}, cargo_nombre={df['cargo_nombre'].unique()}")

2011/presidente: 14718 filas, 1338 mesas, mesa_tipo=['NATIVOS'], cargo_nombre=<StringArray>
['PRESIDENTE/A']
Length: 1, dtype: str
2011/gobernador: 15268 filas, 1388 mesas, mesa_tipo=['NATIVOS'], cargo_nombre=<StringArray>
['GOBERNADOR/A']
Length: 1, dtype: str
2011/intendente: 20006 filas, 1429 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['INTENDENTE']
Length: 1, dtype: str
2015/presidente: 15320 filas, 1532 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['PRESIDENTE']
Length: 1, dtype: str


2015/gobernador: 13788 filas, 1532 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['GOBERNADOR']
Length: 1, dtype: str
2015/intendente: 15320 filas, 1532 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['INTENDENTE']
Length: 1, dtype: str
2019/presidente: 16687 filas, 1517 mesas, mesa_tipo=['NATIVOS'], cargo_nombre=<StringArray>
['PRESIDENTE']
Length: 1, dtype: str
2019/gobernador: 15709 filas, 1594 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['GOBERNADOR']
Length: 1, dtype: str
2019/intendente: 15709 filas, 1594 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['INTENDENTE']
Length: 1, dtype: str


2023/presidente: 16530 filas, 1653 mesas, mesa_tipo=['NATIVOS'], cargo_nombre=<StringArray>
['PRESIDENTE/A']
Length: 1, dtype: str
2023/gobernador: 16245 filas, 1805 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['GOBERNADOR/A']
Length: 1, dtype: str
2023/intendente: 16245 filas, 1805 mesas, mesa_tipo=['EXTRANJEROS', 'NATIVOS'], cargo_nombre=<StringArray>
['INTENDENTE/A']
Length: 1, dtype: str


## 2. Resultado de cada (año, categoría)

In [3]:
for (anio, nombre), df in dataframes.items():
    positivos = (
        df[df["votos_tipo"] == "POSITIVO"]
        .groupby("agrupacion_nombre")["votos_cantidad"]
        .sum()
        .sort_values(ascending=False)
    )
    print(f"=== {anio}/{nombre} ===")
    print(positivos.head(5).to_string())
    print()

=== 2011/presidente ===
agrupacion_nombre
Alianza Frente para la Victoria            153609
Alianza Frente Amplio Progresista           78714
Alianza Unión para el Desarrollo Social     42476
Alianza Compromiso Federal                  30978
Alianza Frente Popular                      27275

=== 2011/gobernador ===
agrupacion_nombre
Alianza Frente para la Victoria            150988
Alianza Unión para el Desarrollo Social     61298
Alianza Frente Amplio Progresista           58629
Alianza Frente Popular                      19745
Nuevo Encuentro                             16923

=== 2011/intendente ===
agrupacion_nombre
Alianza Frente para la Victoria            160079
Alianza Frente Amplio Progresista           45165
Alianza Unión para el Desarrollo Social     37390
Frente Social de la Pcia. Bs.As.            30037
Alianza Frente Popular                      24291

=== 2015/presidente ===
agrupacion_nombre
ALIANZA CAMBIEMOS                                    163852
ALIANZA FRENTE PARA

## 3. Validar contra el agregado de la API (JSON)

Sumamos los positivos del CSV por agrupación y los comparamos contra `client.get_resultados` (el agregado JSON, un solo pedido adicional por categoría). A diferencia de los notebooks anteriores, acá **no asumimos que siempre van a coincidir** — ya sabemos que el agregado JSON puede subestimar (visto antes en CABA/2019, y de nuevo en La Plata/2019/Presidente). El CSV es la fuente confiable; esta validación sirve para detectar en qué (año, categoría) el agregado JSON no se puede usar.

In [4]:
def normalizar_id(x):
    x = str(x)
    return (x.lstrip("0") or "0") if x.isdigit() else x


for anio in ANIOS:
    for nombre, categoria_id in CATEGORIAS.items():
        df = dataframes[(anio, nombre)]
        positivos_csv = (
            df[df["votos_tipo"] == "POSITIVO"]
            .groupby("agrupacion_id")["votos_cantidad"]
            .sum()
        )
        positivos_csv.index = positivos_csv.index.map(lambda x: normalizar_id(x))

        raw = client.get_resultados(
            anio_eleccion=anio, categoria_nombre=f"{nombre}/generales", categoria_id=categoria_id, **LA_PLATA
        )
        resultado = ResultadoElectoral.from_json(raw)
        positivos_api = {
            normalizar_id(a.id_agrupacion): a.votos
            for a in resultado.valores_totalizados_positivos
        }

        agrupaciones = sorted(set(positivos_csv.index) | set(positivos_api))
        diffs = [ag for ag in agrupaciones if positivos_csv.get(ag, 0) != positivos_api.get(ag, 0)]
        estado = "OK" if not diffs else f"AGREGADO JSON NO CONFIABLE (difiere en {len(diffs)} agrupaciones)"
        print(f"{anio}/{nombre}: {estado}")

2011/presidente: OK
2011/gobernador: OK
2011/intendente: OK
2015/presidente: OK
2015/gobernador: OK
2015/intendente: OK
2019/presidente: AGREGADO JSON NO CONFIABLE (difiere en 6 agrupaciones)
2019/gobernador: OK
2019/intendente: OK
2023/presidente: OK
2023/gobernador: OK
2023/intendente: OK


## 4. Análisis mesa por mesa, directo del CSV (ejemplo: La Plata/Presidente/2011)

El CSV ya trae la columna `mesa_id`: alcanza para cualquier análisis mesa por mesa sin pedirle nada más a la API. Ejemplo: participación por mesa y votos del ganador por mesa.

In [5]:
df_2011_presidente = dataframes[(2011, "presidente")]

participacion = df_2011_presidente.groupby("mesa_id").agg(
    votantes=("votos_cantidad", "sum"),
    electores=("mesa_electores", "first"),
    circuito=("circuito_id", "first"),
)
participacion["participacion_pct"] = 100 * participacion["votantes"] / participacion["electores"]

print("mesas con mayor participación:")
print(participacion.sort_values("participacion_pct", ascending=False).head(5))
print("\nmesas con menor participación:")
print(participacion.sort_values("participacion_pct").head(5))

ganador = (
    df_2011_presidente[df_2011_presidente["votos_tipo"] == "POSITIVO"]
    .groupby("agrupacion_nombre")["votos_cantidad"]
    .sum()
    .idxmax()
)
votos_ganador_por_mesa = (
    df_2011_presidente[
        (df_2011_presidente["votos_tipo"] == "POSITIVO")
        & (df_2011_presidente["agrupacion_nombre"] == ganador)
    ]
    .set_index("mesa_id")["votos_cantidad"]
    .sort_index()
)
print(f"\nganador 2011/presidente: {ganador}")
print("votos del ganador, primeras 5 mesas:")
print(votos_ganador_por_mesa.head(5))

mesas con mayor participación:
         votantes  electores circuito  participacion_pct
mesa_id                                                 
614            65         65    0497C         100.000000
916           102        103    0503           99.029126
1116          104        106    0505           98.113208
725           344        351    0501           98.005698
1172          343        350    0508A          98.000000

mesas con menor participación:
         votantes  electores circuito  participacion_pct
mesa_id                                                 
174             4        176    0477            2.272727
846           166        350    0502           47.428571
523           188        351    0496E          53.561254
954           193        349    0504           55.300860
560           214        348    0497           61.494253

ganador 2011/presidente: Alianza Frente para la Victoria
votos del ganador, primeras 5 mesas:
mesa_id
1    81
2    83
3    86
4    57
5   

## 5. Estado final del caché en disco

Cada (año, categoría) debería tener exactamente 2 archivos: el agregado (`.json`) y el CSV oficial (`.csv`). Nada de JSON por mesa acumulado.

In [6]:
ok = True
for anio in ANIOS:
    for nombre in CATEGORIAS:
        archivos = sorted(p.name for p in (REPO / "data" / "distrito" / str(anio) / nombre / "generales").iterdir())
        if len(archivos) != 2:
            ok = False
            print(f"{anio}/{nombre}: ¡{len(archivos)} archivos! {archivos}")

print("OK: todas las carpetas tienen exactamente 2 archivos." if ok else "hay carpetas con archivos de más")

2011/presidente: ¡3 archivos! ['circuito_presidente.json', 'tipoEleccion-2_categoriaId-1_distritoId-2_seccionProvincialId-8_seccionId-63.csv', 'tipoEleccion-2_categoriaId-1_distritoId-2_seccionProvincialId-8_seccionId-63.json']
2011/gobernador: ¡3 archivos! ['circuito_gobernador.json', 'tipoEleccion-2_categoriaId-4_distritoId-2_seccionProvincialId-8_seccionId-63.csv', 'tipoEleccion-2_categoriaId-4_distritoId-2_seccionProvincialId-8_seccionId-63.json']
2011/intendente: ¡3 archivos! ['circuito_intendente.json', 'tipoEleccion-2_categoriaId-7_distritoId-2_seccionProvincialId-8_seccionId-63.csv', 'tipoEleccion-2_categoriaId-7_distritoId-2_seccionProvincialId-8_seccionId-63.json']
2015/presidente: ¡3 archivos! ['circuito_presidente.json', 'tipoEleccion-2_categoriaId-1_distritoId-2_seccionProvincialId-8_seccionId-63.csv', 'tipoEleccion-2_categoriaId-1_distritoId-2_seccionProvincialId-8_seccionId-63.json']
2015/gobernador: ¡3 archivos! ['circuito_gobernador.json', 'tipoEleccion-2_categoriaId-4

## 6. Tabla de agrupaciones por año y nivel

`data/agrupaciones/clasificacion_ideologica_agrupaciones.csv` (compartido con el notebook 03, que agrega los niveles legislativos): una fila por (año, agrupación, nivel). **No se regenera ni se pisa** -- ya tiene una 4ª columna (`campo_ideologico`) clasificada a mano; este paso solo agrega, con aviso explícito, las agrupaciones nuevas que la API todavía no tenía registradas — qué agrupaciones compitieron en cada nivel de cargo en cada elección. Sale del agregado JSON (no del CSV): trae la misma lista de agrupaciones con muchísimo menos texto que parsear (el agregado tiene una fila por agrupación; el CSV tiene una fila por mesa×agrupación×tipo de voto). Verificado que la lista de agrupaciones no se ve afectada por el bug de conteo del agregado JSON (2019/presidente): las agrupaciones que aparecen son las mismas en JSON y en CSV, aunque los votos totales del agregado estén mal — el bug subestima mesas, no hace desaparecer agrupaciones del padrón de listas.

`nivel` usa el nombre que pidieron: `presidente` / `gobernacion` / `intendente` (no `gobernador`).

In [7]:
NIVELES = {"presidente": "presidente", "gobernador": "gobernacion", "intendente": "intendente"}

filas_agrupaciones = []
for anio in ANIOS:
    for nombre, categoria_id in CATEGORIAS.items():
        raw = client.get_resultados(
            anio_eleccion=anio, categoria_nombre=f"{nombre}/generales", categoria_id=categoria_id, **LA_PLATA
        )
        resultado = ResultadoElectoral.from_json(raw)
        for a in resultado.valores_totalizados_positivos:
            filas_agrupaciones.append(
                {"anio": anio, "agrupacion": a.nombre_agrupacion.upper(), "nivel": NIVELES[nombre]}
            )

df_generado = (
    pd.DataFrame(filas_agrupaciones)
    .drop_duplicates()
    .sort_values(["anio", "nivel", "agrupacion"])
    .reset_index(drop=True)
)

# clasificacion_ideologica_agrupaciones.csv (compartido con el notebook 03) tiene
# una 4ª columna (campo_ideologico) clasificada a mano que la API no puede reponer
# — este paso NUNCA sobreescribe el archivo. Solo valida si aparece alguna agrupación
# (anio, nivel, agrupacion) que todavía no esté, la informa, y la agrega con
# campo_ideologico vacío (a clasificar aparte). Las filas de niveles legislativos
# que ya estén en el archivo (agregadas por el notebook 03) se conservan intactas.
destino = REPO / "data" / "agrupaciones"
destino.mkdir(parents=True, exist_ok=True)
archivo = destino / "clasificacion_ideologica_agrupaciones.csv"

if archivo.exists():
    df_existente = pd.read_csv(archivo, keep_default_na=False, dtype={"anio": int})
    clave_existente = set(zip(df_existente["anio"], df_existente["nivel"], df_existente["agrupacion"]))
    es_nueva = df_generado.apply(lambda r: (r["anio"], r["nivel"], r["agrupacion"]) not in clave_existente, axis=1)
    nuevas = df_generado[es_nueva].copy()

    if nuevas.empty:
        print(f"Sin agrupaciones nuevas -- {archivo} no se modifica ({len(df_existente)} filas).")
        df_agrupaciones = df_existente
    else:
        nuevas["campo_ideologico"] = ""
        print(f"AVISO: {len(nuevas)} agrupacion(es) nueva(s), sin clasificar todavia:")
        print(nuevas[["anio", "nivel", "agrupacion"]].to_string(index=False))
        df_agrupaciones = (
            pd.concat([df_existente, nuevas], ignore_index=True)
            .sort_values(["anio", "nivel", "agrupacion"])
            .reset_index(drop=True)
        )
        df_agrupaciones.to_csv(archivo, index=False)
        print(f"Se agregaron {len(nuevas)} fila(s) nueva(s) a {archivo} "
              f"(las {len(df_existente)} existentes no se tocaron).")
else:
    df_generado["campo_ideologico"] = ""
    df_generado.to_csv(archivo, index=False)
    print(f"{archivo} no existia: se creo con {len(df_generado)} filas, todas sin clasificar.")
    df_agrupaciones = df_generado

df_agrupaciones.head(10)

Sin agrupaciones nuevas -- /workspaces/analisis-politica-economia/data/agrupaciones/clasificacion_ideologica_agrupaciones.csv no se modifica (158 filas).


,anio,agrupacion,nivel,campo_ideologico
0,2011,ALIANZA FRENTE AMPLIO PROGRESISTA,gobernacion,2
1,2011,ALIANZA FRENTE DE IZQUIERDA Y DE LOS TRABAJADORES,gobernacion,1
2,2011,ALIANZA FRENTE PARA LA VICTORIA,gobernacion,3
3,2011,ALIANZA FRENTE POPULAR,gobernacion,3
4,2011,ALIANZA PROYECTO SUR,gobernacion,2
5,2011,ALIANZA UNION PARA EL DESARROLLO SOCIAL - UDESO,gobernacion,4
6,2011,ALIANZA UNIÓN PARA EL DESARROLLO SOCIAL,gobernacion,4
7,2011,COALICION CIVICA ARI,gobernacion,4
8,2011,COALICIÓN CÍVICA - AFIRMACIÓN PARA UNA REPÚBLI...,gobernacion,4
9,2011,COMPROMISO FEDERAL,gobernacion,4


## 7. Ampliar etapas: PASO (§2.3 del plan de correcciones)

Mismo cliente y mismos `categoria_id` que Generales (`CATEGORIAS`, arriba) — el
cambio es `tipo_eleccion=1` (PASO) en vez de `2`. El caché va a una subcarpeta
nueva, `data/<año>/<cargo>/paso/` (pasando `categoria_nombre=f"{nombre}/paso"`;
`categoria_nombre` es solo una etiqueta que arma la ruta de caché, no un
parámetro de la API — ver `src/electoral/client.py`). Los archivos de
Generales que ya estaban en `data/<año>/<cargo>/` **no se tocan ni se mueven**
— la reorganización para que Generales también viva en su propia subcarpeta
queda para después.

In [8]:
from electoral.client import ResultadosNoDisponibles

PASO = dict(LA_PLATA)
PASO["tipo_eleccion"] = 1  # PASO

paso_dataframes = {}
paso_faltantes = []
for anio in ANIOS:
    for nombre, categoria_id in CATEGORIAS.items():
        categoria_nombre = f"{nombre}/paso"
        try:
            csv_bytes = client.get_resultados_csv(
                anio_eleccion=anio, categoria_nombre=categoria_nombre, categoria_id=categoria_id, **PASO
            )
        except ResultadosNoDisponibles:
            paso_faltantes.append((anio, nombre))
            continue
        df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
        df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()
        paso_dataframes[(anio, nombre)] = df

        # JSON agregado solo se pide (y se cachea) para los casos que el CSV confirmó
        # disponibles — pedirlo para un caso "no disponible" no tira error (el JSON
        # devuelve 0 mesas en vez de fallar) y terminaría cacheando un archivo vacío.
        raw = client.get_resultados(
            anio_eleccion=anio, categoria_nombre=categoria_nombre, categoria_id=categoria_id, **PASO
        )
        positivos_csv = df.loc[df["votos_tipo"] == "POSITIVO", "votos_cantidad"].sum()
        positivos_json = sum(a["votos"] for a in raw["valoresTotalizadosPositivos"])
        estado = "OK" if positivos_csv == positivos_json else f"DIFERENCIA: {positivos_csv} vs {positivos_json}"
        print(f"PASO {anio}/{nombre}: {len(df)} filas, {df['mesa_id'].nunique()} mesas, "
              f"{df['agrupacion_nombre'].nunique()} agrupaciones, positivos csv vs json -> {estado}")

print("\nPASO no disponible (inesperado si aparece acá: los 4 años ejecutivos tuvieron PASO):")
for anio, nombre in paso_faltantes:
    print(f"  {anio}/{nombre}")

PASO 2011/presidente: 19306 filas, 1379 mesas, 11 agrupaciones, positivos csv vs json -> OK


PASO 2011/gobernador: 20174 filas, 1441 mesas, 10 agrupaciones, positivos csv vs json -> OK


PASO 2015/presidente: 29393 filas, 1547 mesas, 12 agrupaciones, positivos csv vs json -> OK
PASO 2015/gobernador: 26299 filas, 1547 mesas, 11 agrupaciones, positivos csv vs json -> OK


PASO 2015/intendente: 41769 filas, 1547 mesas, 14 agrupaciones, positivos csv vs json -> OK
PASO 2019/presidente: 23335 filas, 1556 mesas, 11 agrupaciones, positivos csv vs json -> OK
PASO 2019/gobernador: 22659 filas, 1636 mesas, 10 agrupaciones, positivos csv vs json -> OK


PASO 2019/intendente: 34110 filas, 1636 mesas, 12 agrupaciones, positivos csv vs json -> OK
PASO 2023/presidente: 46284 filas, 1653 mesas, 16 agrupaciones, positivos csv vs json -> OK


PASO 2023/gobernador: 54150 filas, 1805 mesas, 24 agrupaciones, positivos csv vs json -> OK
PASO 2023/intendente: 61370 filas, 1805 mesas, 23 agrupaciones, positivos csv vs json -> OK

PASO no disponible (inesperado si aparece acá: los 4 años ejecutivos tuvieron PASO):
  2011/intendente


**Sobre `2011/intendente` en `paso_faltantes`**: se investigó si era un problema
de `categoria_id` (probando idCargo 1 a 11 para PASO 2011 completo) y no
aparece ninguna categoría municipal — el CSV devuelve "no disponible"
consistentemente para intendente. La hipótesis más consistente con la ley de
PASO (26.571) es que la interna de Intendente 2011 en La Plata no estuvo
disputada (candidatura única dentro de cada lista), por lo que no se realizó
un comicio primario para esa categoría — pero esto no se pudo confirmar con
una fuente externa, así que queda documentado como caso a revisar, no como
un hecho asumido.

## 8. Ampliar etapas: balotaje / segunda vuelta (§2.3)

Solo aplica a **Presidente**, y solo en los años en que efectivamente hubo
segunda vuelta: **2015** (Macri vs. Scioli) y **2023** (Milei vs. Massa). En
2011 y 2019 el resultado se definió en primera vuelta — verificado pidiendo
`get_resultados_csv` con `tipo_eleccion=3` para esos años: la API devuelve
"no disponible". Gobernador e Intendente tampoco tienen segunda vuelta en la
Provincia de Buenos Aires (se definen por simple pluralidad) — también
verificado de la misma forma. Mismo patrón de caché:
`data/<año>/presidente/balotaje/`.

In [9]:
BALOTAJE = dict(LA_PLATA)
BALOTAJE["tipo_eleccion"] = 3  # Segunda vuelta

ANIOS_CON_BALOTAJE = [2015, 2023]  # 2011 y 2019 se definieron en primera vuelta

balotaje_dataframes = {}
for anio in ANIOS_CON_BALOTAJE:
    categoria_id = CATEGORIAS["presidente"]
    categoria_nombre = "presidente/balotaje"
    csv_bytes = client.get_resultados_csv(
        anio_eleccion=anio, categoria_nombre=categoria_nombre, categoria_id=categoria_id, **BALOTAJE
    )
    df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
    df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()
    balotaje_dataframes[anio] = df

    raw = client.get_resultados(
        anio_eleccion=anio, categoria_nombre=categoria_nombre, categoria_id=categoria_id, **BALOTAJE
    )
    positivos_csv = df.loc[df["votos_tipo"] == "POSITIVO", "votos_cantidad"].sum()
    positivos_json = sum(a["votos"] for a in raw["valoresTotalizadosPositivos"])
    estado = "OK" if positivos_csv == positivos_json else f"DIFERENCIA: {positivos_csv} vs {positivos_json}"
    agrupaciones = sorted(df.loc[df["votos_tipo"] == "POSITIVO", "agrupacion_nombre"].unique())
    print(f"balotaje {anio}/presidente: {len(df)} filas, {df['mesa_id'].nunique()} mesas, "
          f"agrupaciones={agrupaciones}, positivos csv vs json -> {estado}")

# Confirma explícitamente (no solo se asume) que no hay balotaje en los demás casos:
# Presidente 2011/2019 (primera vuelta) y Gobernador/Intendente en los años con balotaje
# presidencial (no hay segunda vuelta para esos cargos en la Provincia de Buenos Aires).
sin_balotaje = [(2011, "presidente", CATEGORIAS["presidente"]), (2019, "presidente", CATEGORIAS["presidente"])] + [
    (anio, nombre, CATEGORIAS[nombre])
    for anio in ANIOS_CON_BALOTAJE
    for nombre in ("gobernador", "intendente")
]
for anio, nombre, categoria_id in sin_balotaje:
    try:
        client.get_resultados_csv(
            anio_eleccion=anio, categoria_nombre=f"{nombre}/balotaje", categoria_id=categoria_id, **BALOTAJE
        )
        print(f"AVISO: {anio}/{nombre} balotaje SÍ está disponible (revisar supuesto)")
    except ResultadosNoDisponibles:
        print(f"{anio}/{nombre}: sin balotaje, confirmado (no disponible)")

balotaje 2015/presidente: 8988 filas, 1498 mesas, agrupaciones=['CAMBIEMOS', 'FRENTE PARA LA VICTORIA'], positivos csv vs json -> OK
balotaje 2023/presidente: 9774 filas, 1629 mesas, agrupaciones=['LA LIBERTAD AVANZA', 'UNION POR LA PATRIA'], positivos csv vs json -> OK
2011/presidente: sin balotaje, confirmado (no disponible)
2019/presidente: sin balotaje, confirmado (no disponible)
2015/gobernador: sin balotaje, confirmado (no disponible)
2015/intendente: sin balotaje, confirmado (no disponible)
2023/gobernador: sin balotaje, confirmado (no disponible)
2023/intendente: sin balotaje, confirmado (no disponible)


## 9. Estado final del caché para PASO y balotaje

Mismo chequeo que la sección 5, ahora sobre las subcarpetas nuevas: cada
`data/<año>/<cargo>/paso/` y `data/<año>/presidente/balotaje/` debería tener
exactamente 2 archivos (`.csv` + `.json`).

In [10]:
ok = True
for anio, nombre in paso_dataframes:
    carpeta = REPO / "data" / "distrito" / str(anio) / nombre / "paso"
    archivos = sorted(p.name for p in carpeta.iterdir())
    if len(archivos) != 2:
        ok = False
        print(f"AVISO: {carpeta} tiene {len(archivos)} archivos: {archivos}")

for anio in balotaje_dataframes:
    carpeta = REPO / "data" / "distrito" / str(anio) / "presidente" / "balotaje"
    archivos = sorted(p.name for p in carpeta.iterdir())
    if len(archivos) != 2:
        ok = False
        print(f"AVISO: {carpeta} tiene {len(archivos)} archivos: {archivos}")

print("OK: todas las carpetas de PASO y balotaje tienen exactamente 2 archivos." if ok else "\nRevisar avisos arriba.")

OK: todas las carpetas de PASO y balotaje tienen exactamente 2 archivos.
